# Notebook 02 — RAG Pipeline

Demonstrates retrieval-augmented generation against the sample program corpus,
including retrieval failure modes and their mitigations.

<!-- TODO main-session: expand teaching framing; tie back to NB 01 closing arc -->

## Setup

Adds the repo root to `sys.path`, loads environment variables, and imports the public RAG API.

<!-- TODO main-session: expand teaching framing -->

In [6]:
from __future__ import annotations
import os, sys, logging
from pathlib import Path

# Silence ChromaDB telemetry noise (posthog API mismatch; does not affect functionality)
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
logging.getLogger("chromadb.telemetry").setLevel(logging.CRITICAL)
logging.getLogger("chromadb.telemetry.product.posthog").setLevel(logging.CRITICAL)
# Silence ChromaDB "requested results > index size" warnings (harmless; Chroma auto-clamps)
logging.getLogger("chromadb").setLevel(logging.ERROR)

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dotenv import load_dotenv
load_dotenv(repo_root / ".env", override=False)

from src.rag import ingest, retrieve, RetrievedDocument
from src.llm import LLMClient

provider = os.getenv("LLM_PROVIDER", "anthropic")
has_key = bool(os.getenv("ANTHROPIC_API_KEY"))
print(f"Provider: {provider} · Anthropic key present: {has_key}")


Provider: anthropic · Anthropic key present: True


## Ingest the corpus

Loads the five sample markdown documents, chunks them, embeds them, and persists the vector store.

<!-- TODO main-session: expand teaching framing -->

In [7]:
corpus_dir = repo_root / "data"
persist_dir = repo_root / "data" / "chroma_nb02"

result = ingest(
    corpus_dir=corpus_dir,
    persist_dir=persist_dir,
    chunk_size=500,
    chunk_overlap=50,
)

print(f"documents_loaded  : {result.documents_loaded}")
print(f"chunks_created    : {result.chunks_created}")
print(f"chunks_indexed    : {result.chunks_indexed}")
print(f"vector_store_path : {result.vector_store_path}")
print(f"embedding_model   : {result.embedding_model}")

documents_loaded  : 5
chunks_created    : 42
chunks_indexed    : 42
vector_store_path : c:\Users\narla\OneDrive\Desktop\TalentSprint\IISc_GenAI_C2\LLMOps\llmops-session\data\chroma_nb02
embedding_model   : sentence-transformers/all-MiniLM-L6-v2


## Baseline retrieval

Retrieve the top-5 chunks for a simple policy question and inspect scores + source priority.

<!-- TODO main-session: expand teaching framing -->


In [6]:
hits = retrieve(persist_dir, "What is the late submission policy?", k=5)

SEP = "-" * 60
for i, doc in enumerate(hits, 1):
    doc_id = doc.chunk.metadata.document_id
    priority = doc.chunk.metadata.source_priority
    print(f"Hit {i}")
    print(f"  document_id     : {doc_id}")
    print(f"  source_priority : {priority}")
    print(f"  score           : {doc.score:.4f}")
    print(f"  text (first 200): {doc.chunk.text[:200]!r}")
    print(SEP)


Hit 1
  document_id     : program_policy
  source_priority : 1
  score           : 0.9911
  text (first 200): '- Submissions up to **48 hours late** receive full credit with no\n  penalty if a brief note is added to the submission explaining the delay.\n- Submissions **48–168 hours late** (i.e. up to one week) r'
------------------------------------------------------------
Hit 2
  document_id     : assignment_guidelines
  source_priority : 2
  score           : 0.9769
  text (first 200): '## Late Submission Handling\n\nSee the **Program Policy** document, section *Late Submission Policy*, for\nthe authoritative rules. In summary: 48 hours grace with note → 10% penalty\nup to one week → not'
------------------------------------------------------------
Hit 3
  document_id     : faq
  source_priority : 5
  score           : 0.9441
  text (first 200): '**Q: What file format?**\nWhatever the assignment brief specifies. Default is `.ipynb` with outputs\nsaved for notebook assignments.\n\n**Q

## Notice the source_priority

The policy doc (`source_priority=1`) ranks at or near the top for a policy question — the retriever naturally surfaces the authoritative source. Later sections show when this breaks down.

<!-- TODO main-session: expand teaching framing -->


## Failure 1: parametric knowledge is not your policy

Cell 9 asks about a common program rule with no context — the model hedges because it correctly knows it doesn't have your specific policy. Cell 10 retrieves the exact rule. Vague training-data knowledge is not actionable; retrieved facts are.

<!-- TODO main-session: expand teaching framing -->

In [16]:
# Cell 9 — UNGROUNDED: ask about attendance norms with no retrieved context.
# Asking for the "most common" value forces a single number from parametric knowledge.
# Industry norm is typically 75%; our corpus sets 80% — close but wrong.

QUESTION = (
    "In a typical online AI engineering certification program, "
    "what is the standard minimum percentage of live sessions "
    "a participant must attend to remain in good standing?"
)

if not os.getenv("ANTHROPIC_API_KEY"):
    print("[skipped — no ANTHROPIC_API_KEY]")
else:
    client = LLMClient()
    result_ungrounded = client.complete(
        prompt=QUESTION,
        system=(
            "You are a helpful assistant. "
            "If the exact value varies by provider, give the single most commonly used figure."
        ),
        cache=False,
    )
    print("=== PARAMETRIC (no retrieval) ===")
    print(result_ungrounded.text)
    print(f"\n(This program's actual policy: 80% — see cell 10)")


=== PARAMETRIC (no retrieval) ===
I don't have reliable data on a universally "standard" minimum attendance requirement for online AI engineering certification programs, as this varies significantly by provider.

However, based on common practices across online professional certifications:

- **60-80% attendance** is a typical range for programs that track live session participation
- Some programs use **75% as a benchmark**
- Others are more flexible, requiring attendance only for specific mandatory sessions rather than all live sessions

**Important caveats:**
- Many online certification programs don't mandate live attendance at all—they offer recordings and asynchronous content
- Requirements differ between providers (Coursera, edX, university programs, corporate platforms, etc.)
- "Good standing" definitions vary (some focus on assignments/assessments rather than attendance)

If you're evaluating a specific program, I'd recommend checking their:
- Student handbook or program requir

In [17]:
# Cell 10 — GROUNDED: retrieve the attendance policy chunk, then ask the same question.
# The retrieved chunk explicitly states 80%; the grounded answer matches the doc exactly.

ATTENDANCE_QUESTION = (
    "What is the minimum percentage of live sessions a participant must attend "
    "to remain in good standing?"
)

grounded_hits = retrieve(persist_dir, "minimum attendance percentage live sessions", k=3)

context_parts = []
for rank, doc in enumerate(grounded_hits, 1):
    label = (
        f"[Source {rank}: {doc.chunk.metadata.document_id} "
        f"(priority={doc.chunk.metadata.source_priority}, score={doc.score:.4f})]"
    )
    context_parts.append(f"{label}\n{doc.chunk.text}")

context_block = "\n\n".join(context_parts)

grounded_prompt = (
    f"Use only the documents provided below to answer the question.\n\n"
    f"{context_block}\n\n"
    f"Question: {ATTENDANCE_QUESTION}"
)

print("=== RETRIEVED CONTEXT ===")
for part in context_parts:
    print(part[:300])
    print(SEP)

if not os.getenv("ANTHROPIC_API_KEY"):
    print("[skipped — no ANTHROPIC_API_KEY]")
else:
    client = LLMClient()
    result_grounded = client.complete(
        prompt=grounded_prompt,
        system=(
            "You are a helpful assistant. Answer only from the provided documents. "
            "If the answer is not in the documents, say so explicitly."
        ),
        cache=False,
    )
    print("\n=== GROUNDED ANSWER ===")
    print(result_grounded.text)


=== RETRIEVED CONTEXT ===
[Source 1: program_policy (priority=1, score=0.9800)]
## Attendance Expectations

Participants are expected to attend a minimum of **80% of live sessions**.
Attendance is tracked automatically by the learning platform.
------------------------------------------------------------
[Source 2: schedule (priority=4, score=0.9016)]
# Sample Program Schedule

> Synthetic document used for the LLM Ops teaching session. Dates and times
> are illustrative and do not refer to any real cohort.

## Weekly Session Pattern

| Day | Time (IST) | Session type |
|---|---|---|
| Tuesday | 19:
------------------------------------------------------------
[Source 3: faq (priority=5, score=0.8234)]
**Q: I joined late — can I still attend?**
Yes. Late joining is fine. Attendance is counted as long as you join within
the first 30 minutes of the session.

**Q: Can I attend on mobile?**
Theory sessions work fine on mobile. Hands-on sessions are not recomm
-------------------------------

## What just happened

Without retrieval the model hedged: *"60–80%, commonly 75%, check your handbook."* With retrieval it gave the exact rule: **80% of live sessions, tracked automatically.** A participant relying on the hedged answer might assume 75% and lose their standing. RAG replaces vague averages with authoritative facts.

<!-- TODO main-session: expand teaching framing -->

## Failure 2: wrong doc on top

Raw similarity retrieval picks whatever text is closest to the query — not whatever source is most authoritative.

<!-- TODO main-session: expand teaching framing -->


In [26]:
# Cell 13 — raw retrieval: wrong doc on top
# A query about assignment deadlines is lexically closest to the assignment_guidelines
# doc (source_priority=2), which edges out the authoritative program_policy (source_priority=1).
# The guidelines doc even defers to policy for the authoritative rules — but retrieval
# doesn't know that.

BAD_QUERY = "when is assignment 2 due"
SEP2 = "-" * 64

print(f"Query : {BAD_QUERY!r}  (k=3, raw similarity order)")
print(SEP2)
bad_hits = retrieve(persist_dir, BAD_QUERY, k=3)
for rank, doc in enumerate(bad_hits, 1):
    doc_id   = doc.chunk.metadata.document_id
    priority = doc.chunk.metadata.source_priority
    snippet  = doc.chunk.text[:160].replace("\n", " ")
    print(f"#{rank}  document_id={doc_id!r:28s}  source_priority={priority}  score={doc.score:.4f}")
    print(f"     {snippet!r}")
    print()

priorities_returned = [d.chunk.metadata.source_priority for d in bad_hits]
top_priority        = priorities_returned[0]
policy_in_results   = 1 in priorities_returned
if not policy_in_results:
    print(f"⚠  Policy doc (priority=1) is ABSENT from k=3 results — buried below lower-priority docs.")
elif top_priority > 1:
    print(f"⚠  Top result has source_priority={top_priority} — policy doc (priority=1) is NOT first.")
else:
    print(f"✓  Policy doc (priority=1) is already first — no failure to show; tune the query.")


Query : 'when is assignment 2 due'  (k=3, raw similarity order)
----------------------------------------------------------------
#1  document_id='assignment_guidelines'       source_priority=2  score=0.9556
     '# Sample Assignment Guidelines  > Synthetic document used for the LLM Ops teaching session.  ## Assignment Objective  Weekly assignments reinforce the concepts '

#2  document_id='support_process'             source_priority=3  score=0.8905
     '## Academic Doubt Process  1. Re-read the assignment brief and rubric 2. Check the FAQ document 3. If still unclear, post in `#academic-help` with a specific qu'

#3  document_id='faq'                         source_priority=5  score=0.8221
     '## Peer Discussion Rules  **Q: Can I discuss assignments with peers?** Discussing concepts is fine and encouraged. Sharing code or solutions is not permitted un'

⚠  Policy doc (priority=1) is ABSENT from k=3 results — buried below lower-priority docs.


In [29]:
# Cell 14 — fix: retrieve k=10, re-rank by source_priority, take top 3
# Fetching more candidates gives us the full authority ladder.
# Sorting by source_priority (lower = more authoritative) promotes the policy doc to #1.

print(f"Query : {BAD_QUERY!r}  (k=10 → sorted by source_priority asc)")
print(SEP2)
wide_hits  = retrieve(persist_dir, BAD_QUERY, k=10)
reranked   = sorted(wide_hits, key=lambda d: d.chunk.metadata.source_priority)
top3       = reranked[:3]

for rank, doc in enumerate(top3, 1):
    doc_id   = doc.chunk.metadata.document_id
    priority = doc.chunk.metadata.source_priority
    snippet  = doc.chunk.text[:160].replace("\n", " ")
    print(f"#{rank}  document_id={doc_id!r:28s}  source_priority={priority}  score={doc.score:.4f}")
    print(f"     {snippet!r}")
    print()

policy_rank = next(
    (i + 1 for i, d in enumerate(top3) if d.chunk.metadata.source_priority == 1), None
)
print(f"✓  Policy doc (source_priority=1) is now #{policy_rank} after re-ranking.")


Query : 'when is assignment 2 due'  (k=10 → sorted by source_priority asc)
----------------------------------------------------------------
#1  document_id='program_policy'              source_priority=1  score=0.9992
     '## Late Submission Policy  Assignments are due by **23:59 IST on the stated due date**.'

#2  document_id='assignment_guidelines'       source_priority=2  score=0.9903
     '# Sample Assignment Guidelines  > Synthetic document used for the LLM Ops teaching session.  ## Assignment Objective  Weekly assignments reinforce the concepts '

#3  document_id='support_process'             source_priority=3  score=0.9828
     '## Academic Doubt Process  1. Re-read the assignment brief and rubric 2. Check the FAQ document 3. If still unclear, post in `#academic-help` with a specific qu'

✓  Policy doc (source_priority=1) is now #1 after re-ranking.


## Source priority is the second-line defense

Re-ranking by `source_priority` costs one sort; it ensures policy-level documents are always promoted above summaries and FAQs even when embedding similarity disagrees.

<!-- TODO main-session: expand teaching framing -->


## Failure 3: missing context

Cell below retrieves a query the corpus has no answer for, to observe low similarity scores.

<!-- TODO main-session: expand teaching framing -->

In [34]:
# Failure 3: retrieve a query for something genuinely absent from the corpus.
# The corpus covers: program policy, schedule, assignment guidelines, support process, FAQ.
#
# Score note: retrieve() converts ChromaDB cosine-distance d → score = 1/(1+d).
# In-domain queries score ≈ 0.97–0.99 (d ≈ 0.01–0.03).
# Truly out-of-domain queries score ≈ 0.70–0.90 (d ≈ 0.11–0.43).
# Absolute scores < 0.5 require cosine similarity < 0 (uncommon); the contrast
# in-domain (0.99) vs. out-of-domain (≈0.88) is the meaningful signal.

MISSING_QUERY = "What is the recipe for chocolate chip cookies?"
SEP3 = "-" * 60

missing_hits = retrieve(persist_dir, MISSING_QUERY, k=3)

print(f"Query: {MISSING_QUERY!r}\n{SEP3}")
for rank, doc in enumerate(missing_hits, start=1):
    meta = doc.chunk.metadata
    snippet = doc.chunk.text[:200].replace("\n", " ")
    print(
        f"#{rank}  doc={meta.document_id}  priority={meta.source_priority}  score={doc.score:.4f}\n"
        f"     {snippet!r}\n"
    )

max_score = max(d.score for d in missing_hits) if missing_hits else 0.0
print(f"{SEP3}\nMax similarity score: {max_score:.4f}")
print("→ Compare to in-domain queries (≈0.99): this score is noticeably lower.")
print("  A threshold calibrated to this corpus (e.g. 0.93) would trigger refusal.")

Query: 'What is the recipe for chocolate chip cookies?'
------------------------------------------------------------
#1  doc=schedule  priority=4  score=0.8832
     '# Sample Program Schedule  > Synthetic document used for the LLM Ops teaching session. Dates and times > are illustrative and do not refer to any real cohort.  ## Weekly Session Pattern  | Day | Time '

#2  doc=program_policy  priority=1  score=0.7279
     '## Late Submission Policy  Assignments are due by **23:59 IST on the stated due date**.'

#3  doc=assignment_guidelines  priority=2  score=0.7256
     '# Sample Assignment Guidelines  > Synthetic document used for the LLM Ops teaching session.  ## Assignment Objective  Weekly assignments reinforce the concepts introduced in live sessions. Each assign'

------------------------------------------------------------
Max similarity score: 0.8832
→ Compare to in-domain queries (≈0.99): this score is noticeably lower.
  A threshold calibrated to this corpus (e.g. 0.93) would t

In [35]:
# Refusal helper: returns a canned refusal when max retrieval score < min_score.
# Uses LLMClient(provider="mock") for the grounded branch —
# the demo is about the refusal logic, not the LLM answer quality.
#
# min_score calibration note:
#   - In-domain queries score ≈ 0.97–0.99; out-of-domain ≈ 0.70–0.90.
#   - Default 0.4 (the brief's suggestion) would never fire for this score formula;
#     0.93 separates "relevant" from "not covered" cleanly for this corpus.

def grounded_answer_or_refuse(query: str, k: int = 3, min_score: float = 0.4) -> str:
    docs = retrieve(persist_dir, query, k=k)
    if not docs or max(d.score for d in docs) < min_score:
        return (
            "I don't see information about that in the program documents. "
            "Please contact the program team."
        )
    # Build context from retrieved chunks (grounded branch)
    context = "\n\n".join(
        f"[{doc.chunk.metadata.document_id}]\n{doc.chunk.text}" for doc in docs
    )
    prompt = (
        "Answer the question using only the context below.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}"
    )
    mock_client = LLMClient(provider="mock")
    result = mock_client.complete(prompt)
    return result.text


# Calibrated threshold for this corpus's 1/(1+cosine_dist) scoring
CORPUS_THRESHOLD = 0.93

# --- Demo 1: missing-context query → refusal ---
answer_missing = grounded_answer_or_refuse(MISSING_QUERY, k=3, min_score=CORPUS_THRESHOLD)
print(f"Query : {MISSING_QUERY!r}")
print(f"Answer: {answer_missing}")
print()

# --- Demo 2: known-good query → passes threshold → grounded answer (mock) ---
KNOWN_QUERY = "What is the late submission policy?"
answer_known = grounded_answer_or_refuse(KNOWN_QUERY, k=3, min_score=CORPUS_THRESHOLD)
print(f"Query : {KNOWN_QUERY!r}")
print(f"Answer: {answer_known}")

Query : 'What is the recipe for chocolate chip cookies?'
Answer: I don't see information about that in the program documents. Please contact the program team.

Query : 'What is the late submission policy?'
Answer: [mock:4296b078] echo: Answer the question using only the context below...


## Failure 4a: conflicting sources

Multiple documents address the same topic — retrieval surfaces both, but they may disagree on the answer.

<!-- TODO main-session: expand teaching framing -->


In [3]:
# Cell 20: retrieve a query that surfaces BOTH the policy doc (priority=1)
# and the FAQ doc (priority=5).  Both address recording retention; when they
# disagree a reader can't tell which to trust — source_priority resolves that.

CONFLICT_QUERY = "how long are session recordings kept"

hits = retrieve(persist_dir, CONFLICT_QUERY, k=5)

SEP = "-" * 60
print(f"Query: {CONFLICT_QUERY!r}  (k=5)\n")
for i, doc in enumerate(hits, 1):
    doc_id   = doc.chunk.metadata.document_id
    priority = doc.chunk.metadata.source_priority
    print(f"Hit {i}  |  {doc_id:<30}  |  priority={priority}  |  score={doc.score:.4f}")
    print(f"  {doc.chunk.text[:180].strip()!r}")
    print(SEP)


Query: 'how long are session recordings kept'  (k=5)

Hit 1  |  program_policy                  |  priority=1  |  score=0.9877
  "## Session Recording Access\n\nRecordings of live sessions are made available to enrolled participants\nthrough the learning platform within 24 hours of the session's end. Recordings"
------------------------------------------------------------
Hit 2  |  schedule                        |  priority=4  |  score=0.9436
  '- 30 minutes — recap and motivation\n- 60 minutes — core content delivery with worked examples\n- 20 minutes — open Q&A\n- 10 minutes — preview of the hands-on session\n\nTheory session'
------------------------------------------------------------
Hit 3  |  faq                             |  priority=5  |  score=0.9155
  '**Q: How long are recordings kept?**\nFor the duration of the program plus 90 days after program end. See the\n**Program Policy** for the authoritative version.\n\n**Q: Can I share rec'
---------------------------------------

In [4]:
# Cell 21: priority-based resolution.
# Keep only chunks from the single most-authoritative source
# (lowest source_priority value).  The policy doc always wins.

min_priority = min(doc.chunk.metadata.source_priority for doc in hits)
authoritative = [doc for doc in hits if doc.chunk.metadata.source_priority == min_priority]

SEP = "-" * 60
print(f"Query: {CONFLICT_QUERY!r}")
print(f"After priority filter — keeping priority={min_priority} only:\n")
for i, doc in enumerate(authoritative, 1):
    doc_id = doc.chunk.metadata.document_id
    print(f"Hit {i}  |  {doc_id:<30}  |  priority={doc.chunk.metadata.source_priority}  |  score={doc.score:.4f}")
    print(f"  {doc.chunk.text[:200].strip()!r}")
    print(SEP)


Query: 'how long are session recordings kept'
After priority filter — keeping priority=1 only:

Hit 1  |  program_policy                  |  priority=1  |  score=0.9877
  "## Session Recording Access\n\nRecordings of live sessions are made available to enrolled participants\nthrough the learning platform within 24 hours of the session's end. Recordings\nremain accessible fo"
------------------------------------------------------------


## Failure 4b: chunk size matters

Re-ingesting with chunk_size=2000 (too broad) and chunk_size=100 (too narrow) shows how chunking shapes what the retriever can return.

<!-- TODO main-session: expand teaching framing -->


In [5]:
# Cell 23: re-ingest the corpus at two extreme chunk sizes.
# tmp_ prefix keeps these dirs from clashing with chroma_nb02.
# Both temp stores are removed at the end of the cell.

import shutil

CHUNK_QUERY = "What happens to assignments submitted after the deadline?"

tmp_large = repo_root / "data" / "tmp_chroma_chunk2000"
tmp_small  = repo_root / "data" / "tmp_chroma_chunk100"

# Ingest at chunk_size=2000 (each chunk spans multiple policy sections)
print("Ingesting chunk_size=2000 …", end=" ", flush=True)
ingest(corpus_dir=corpus_dir, persist_dir=tmp_large, chunk_size=2000, chunk_overlap=200)
print("done")

# Ingest at chunk_size=100 (chunks are sentence fragments)
print("Ingesting chunk_size=100  …", end=" ", flush=True)
ingest(corpus_dir=corpus_dir, persist_dir=tmp_small, chunk_size=100,  chunk_overlap=10)
print("done")

SEP_WIDE  = "=" * 70
SEP_LIGHT = "-" * 70

print(f"\nQuery: {CHUNK_QUERY!r}\n")

# --- Large chunks ---
print(f"{'  chunk_size=2000 (too broad)  ':=^70}")
for i, doc in enumerate(retrieve(tmp_large, CHUNK_QUERY, k=3), 1):
    chars = len(doc.chunk.text)
    print(f"Hit {i} | {doc.chunk.metadata.document_id:<26} | score={doc.score:.4f} | {chars} chars in chunk")
    print(f"  preview: {doc.chunk.text[:220].strip()!r}")
    print(SEP_LIGHT)

print()

# --- Small chunks ---
print(f"{'  chunk_size=100 (too narrow)  ':=^70}")
for i, doc in enumerate(retrieve(tmp_small, CHUNK_QUERY, k=3), 1):
    chars = len(doc.chunk.text)
    print(f"Hit {i} | {doc.chunk.metadata.document_id:<26} | score={doc.score:.4f} | {chars} chars in chunk")
    print(f"  preview: {doc.chunk.text[:220].strip()!r}")
    print(SEP_LIGHT)

# Cleanup temporary vector stores (do not leave them in the repo)
shutil.rmtree(tmp_large, ignore_errors=True)
shutil.rmtree(tmp_small, ignore_errors=True)
print("\nTemporary persist dirs removed.")


Ingesting chunk_size=2000 … done
Ingesting chunk_size=100  … done

Query: 'What happens to assignments submitted after the deadline?'

===================  chunk_size=2000 (too broad)  ====================
Hit 1 | program_policy             | score=0.7956 | 1757 chars in chunk
  preview: '# Sample Program Policy\n\n> Synthetic document used for the LLM Ops teaching session. Any resemblance\n> to an actual training program is coincidental.\n\n## Program Overview\n\nThe Sample GenAI Program is a 12-week part-time'
----------------------------------------------------------------------
Hit 2 | assignment_guidelines      | score=0.7381 | 1054 chars in chunk
  preview: '## Evaluation Rubric\n\nAssignments are graded against four criteria, each contributing 25%:\n\n| Criterion | What we look for |\n|---|---|\n| Correctness | The deliverable works as specified |\n| Clarity | Code is readable; wr'
----------------------------------------------------------------------
Hit 3 | faq                

## Tuning is unavoidable

The default (chunk_size=500) balanced specificity and context — neither extreme works out of the box.

<!-- TODO main-session: this is a teaching moment about RAG tuning being a real engineering problem, not a configuration step -->


## Failure 5: stale ingestion

Old ingestion returns yesterday's rules — a changed policy is invisible until you re-ingest.

<!-- TODO main-session: expand teaching framing -->


In [12]:
import shutil, textwrap

STALE_QUERY = "What happens to late submissions?"

# ── 1. Read the current (v1) policy ──────────────────────────────────────────
policy_src = corpus_dir / "sample_program_policy.md"
policy_v1_text = policy_src.read_text(encoding="utf-8")

# ── 2. Create a modified v2 in a temp dir ────────────────────────────────────
tmp_corpus_v2 = repo_root / "data" / "tmp_policy_v2"
tmp_corpus_v2.mkdir(exist_ok=True)

# Copy the other 4 docs unchanged
for f in corpus_dir.glob("sample_*.md"):
    if f.name != "sample_program_policy.md":
        shutil.copy(f, tmp_corpus_v2 / f.name)

# Inject the 2026-Q3 update by replacing the first bullet in the Late Submission section.
# This ensures the change lands in the same high-relevance chunk as the original rule.
OLD_RULE = (
    "- Submissions up to **48 hours late** receive full credit with no\n"
    "  penalty if a brief note is added to the submission explaining the delay."
)
NEW_RULE = (
    "> **Update 2026-Q3:** Late submissions now incur a **5-point penalty per day**.\n"
    "> The previous 48-hour no-penalty window was removed effective 2026-07-01.\n"
    "\n"
    "- Submissions up to **48 hours late** now incur a 5-point-per-day deduction"
    " (2026-Q3 policy change)."
)
assert OLD_RULE in policy_v1_text, "Anchor text not found — brief corpus may have changed"
policy_v2_text = policy_v1_text.replace(OLD_RULE, NEW_RULE, 1)
(tmp_corpus_v2 / "sample_program_policy.md").write_text(policy_v2_text, encoding="utf-8")

# ── 3. Ingest the modified corpus into a fresh persist dir ───────────────────
persist_stale_new = repo_root / "data" / "chroma_stale_new"
_ = ingest(corpus_dir=tmp_corpus_v2, persist_dir=persist_stale_new)

# ── 4. Retrieve top result from OLD and NEW vector stores ────────────────────
hit_old = retrieve(persist_dir, STALE_QUERY, k=1)[0]
hit_new = retrieve(persist_stale_new, STALE_QUERY, k=1)[0]

print(f"Query: {STALE_QUERY!r}")
print()
print("=== OLD INGESTION (v1 policy — no per-day penalty) ===")
print(f"  doc: {hit_old.chunk.metadata.document_id}  score: {hit_old.score:.4f}")
print(textwrap.fill(hit_old.chunk.text, width=80, initial_indent="  ", subsequent_indent="  "))

print()
print("=== NEW INGESTION (v2 policy — 2026-Q3 update ingested) ===")
print(f"  doc: {hit_new.chunk.metadata.document_id}  score: {hit_new.score:.4f}")
print(textwrap.fill(hit_new.chunk.text, width=80, initial_indent="  ", subsequent_indent="  "))

# ── 5. Cleanup — actual data/sample_*.md files are untouched ─────────────────
shutil.rmtree(tmp_corpus_v2, ignore_errors=True)
shutil.rmtree(persist_stale_new, ignore_errors=True)
print("\nTemp dirs removed. Actual data/sample_*.md files are unchanged.")


Query: 'What happens to late submissions?'

=== OLD INGESTION (v1 policy — no per-day penalty) ===
  doc: program_policy  score: 0.8865
  - Submissions up to **48 hours late** receive full credit with no   penalty if
  a brief note is added to the submission explaining the delay. - Submissions
  **48–168 hours late** (i.e. up to one week) receive a   10% mark deduction. -
  Submissions more than **one week late** are not accepted except where   a
  documented medical or personal emergency has been raised through   the support
  process. - There is **no automatic 3-day grace period.** If you read that

=== NEW INGESTION (v2 policy — 2026-Q3 update ingested) ===
  doc: program_policy  score: 0.8796
  - Submissions up to **48 hours late** now incur a 5-point-per-day deduction
  (2026-Q3 policy change). - Submissions **48–168 hours late** (i.e. up to one
  week) receive a   10% mark deduction. - Submissions more than **one week
  late** are not accepted except where   a documented medical 

## Putting it all together

NB 02 showed five retrieval failure modes that need handling. But who decides which question gets routed to retrieve, which gets refused, which gets escalated? NB 03 builds that orchestration layer.

<!-- TODO main-session: expand teaching framing; bridge to NB 03 workflow -->


In [13]:
# Closing recap: full best-practice RAG pipeline on one realistic participant question.
# Same question as NB 01 cell 24 — now with real retrieval instead of a hand-pasted doc.

RECAP_QUERY = "can I get extra time on assignment 2 if I'm sick?"
CORPUS_THRESHOLD = 0.93  # separates in-domain (0.97–0.99) from out-of-domain (0.70–0.90)

# ── Step 1: Retrieve k=5 ─────────────────────────────────────────────────────
hits = retrieve(persist_dir, RECAP_QUERY, k=5)

# ── Step 2: Filter — drop chunks below corpus threshold ──────────────────────
hits = [h for h in hits if h.score >= CORPUS_THRESHOLD]

# ── Step 3: Re-rank by source_priority (lower = more authoritative) ──────────
hits_sorted = sorted(hits, key=lambda h: (h.chunk.metadata.source_priority, -h.score))
top_hits = hits_sorted[:3]  # take top 3 by authority

print(f"Query: {RECAP_QUERY!r}")
print()
print("Retrieved & re-ranked sources:")
for i, h in enumerate(top_hits, 1):
    print(
        f"  [{i}] {h.chunk.metadata.document_id}"
        f"  priority={h.chunk.metadata.source_priority}"
        f"  score={h.score:.4f}"
    )
print()

# ── Step 4: Build grounded context ───────────────────────────────────────────
if not top_hits:
    print("No sufficiently relevant documents found. Cannot answer grounded.")
else:
    context = "\n\n".join(
        f"[source: {h.chunk.metadata.document_id}]\n{h.chunk.text}"
        for h in top_hits
    )
    prompt = (
        "You are a program assistant. Using ONLY the program documents provided below, "
        "answer the participant's question. If the answer is not in the documents, say so.\n\n"
        f"{context}\n\n"
        f"Question: {RECAP_QUERY}\n"
        "Answer:"
    )

    # ── Step 5: Generate answer ───────────────────────────────────────────────
    if has_key:
        client = LLMClient()
        answer = client.complete(prompt)
    else:
        answer = "[skipped — no ANTHROPIC_API_KEY; run with key to see grounded answer]"

    print("Answer:")
    print(answer)


Query: "can I get extra time on assignment 2 if I'm sick?"

Retrieved & re-ranked sources:
  [1] program_policy  priority=1  score=0.9824
  [2] assignment_guidelines  priority=2  score=0.9567

Answer:
CompletionResult(text="Based on the program documents provided, I don't have information about extensions or accommodations for illness or other circumstances.\n\nTo get an answer about extra time due to sickness, you would need to contact your lead instructor or Program Operations directly, as this policy is not covered in the documents I have access to.", model='claude-haiku-4-5-20251001', provider='anthropic', latency_ms=1697.772099985741, tokens_in=263, tokens_out=66, cost_estimate_usd=0.000593, cache_status='miss', raw={'id': 'msg_012F1WSvLFf2Kuiws7EiF9Yf', 'container': None, 'content': [{'citations': None, 'text': "Based on the program documents provided, I don't have information about extensions or accommodations for illness or other circumstances.\n\nTo get an answer about extra t